# 03 — ResolVI-corrected pyUCell scoring and target-compartment exploration

This notebook loads one saved sample-level ResolVI model at a time. It obtains
all-gene corrected expression in cell chunks, immediately converts each chunk
to pyUCell scores, and discards the dense corrected expression chunk. This is
mathematically appropriate for UCell because each cell is ranked independently
and avoids materializing a prohibitively large corrected matrix.

The notebook computes broad lineage signatures, multi-gene coherence counts,
`target_score`, `exclusion_score`, and `target_margin`; writes scored H5AD files;
and produces distributions, spatial score maps, and threshold-grid CSV files.

The final gate is intentionally separate. After reviewing the plots, fill the
`SAMPLE_THRESHOLDS` dictionary and run only the last gate cell to save the
immune/endothelial-enriched objects. High-target/high-exclusion cells are
retained as `target_ambiguous_mixed` by default rather than discarded.


In [ ]:
# ---------------------------------------------------------------------
# GPU selection — run before importing torch/scvi
# ---------------------------------------------------------------------
import os

GPU_ID = '0'
os.environ['CUDA_VISIBLE_DEVICES'] = GPU_ID
os.environ.setdefault('OMP_NUM_THREADS', '16')
os.environ.setdefault('MKL_NUM_THREADS', '16')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '16')
print('CUDA_VISIBLE_DEVICES =', os.environ['CUDA_VISIBLE_DEVICES'])


In [ ]:
from __future__ import annotations

import gc
import json
import time
import warnings
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyro
import pyucell as uc
import scipy.sparse as sp
import torch
from packaging.version import Version

import scvi
from scvi.external import RESOLVI

SAMPLE_INFO = {
    "Screen_39_21": {
        "patient": "patient_39_21",
        "cancer_type": "NSCLC",
        "biopsy_stage": "Screen"
    },
    "C2D15_39_21": {
        "patient": "patient_39_21",
        "cancer_type": "NSCLC",
        "biopsy_stage": "C2D15"
    },
    "Screen_17_26": {
        "patient": "patient_17_26",
        "cancer_type": "NSCLC",
        "biopsy_stage": "Screen"
    },
    "C2D15_17_26": {
        "patient": "patient_17_26",
        "cancer_type": "NSCLC",
        "biopsy_stage": "C2D15"
    },
    "Screen_18_23": {
        "patient": "patient_18_23",
        "cancer_type": "melanoma",
        "biopsy_stage": "Screen"
    },
    "C2D15_18_23": {
        "patient": "patient_18_23",
        "cancer_type": "melanoma",
        "biopsy_stage": "C2D15"
    },
    "Screen_16_22": {
        "patient": "patient_16_22",
        "cancer_type": "melanoma",
        "biopsy_stage": "Screen"
    },
    "C2D15_16_22": {
        "patient": "patient_16_22",
        "cancer_type": "melanoma",
        "biopsy_stage": "C2D15"
    },
    "Screen_30_16": {
        "patient": "patient_30_16",
        "cancer_type": "melanoma",
        "biopsy_stage": "Screen"
    },
    "C2D15_30_16": {
        "patient": "patient_30_16",
        "cancer_type": "melanoma",
        "biopsy_stage": "C2D15"
    },
    "Screen_23_25": {
        "patient": "patient_23_25",
        "cancer_type": "colon_cancer",
        "biopsy_stage": "Screen"
    },
    "C2D15_23_25": {
        "patient": "patient_23_25",
        "cancer_type": "colon_cancer",
        "biopsy_stage": "C2D15"
    }
}
DEFAULT_SIGNATURES = {
    "immune_core": [
        "PTPRC",
        "CD3D",
        "CD3E",
        "TRAC",
        "LCK",
        "IL32",
        "LST1",
        "TYROBP",
        "FCER1G",
        "MS4A1",
        "CD79A",
        "NKG7",
        "KLRD1",
        "AIF1",
        "CTSS"
    ],
    "T_cell": [
        "CD3D",
        "CD3E",
        "TRAC",
        "LCK",
        "CD247",
        "IL32",
        "LTB"
    ],
    "CD4_helper": [
        "IL7R",
        "LTB",
        "CCR7",
        "MAL",
        "TCF7",
        "LEF1"
    ],
    "CD8_T": [
        "CD8A",
        "CD8B",
        "CTSW",
        "CCL5",
        "TRAC",
        "LCK"
    ],
    "Treg": [
        "FOXP3",
        "IL2RA",
        "CTLA4",
        "TIGIT",
        "IKZF2",
        "TNFRSF18"
    ],
    "NK": [
        "KLRD1",
        "NKG7",
        "GNLY",
        "PRF1",
        "FCER1G",
        "TYROBP",
        "TRDC"
    ],
    "B_cell": [
        "MS4A1",
        "CD79A",
        "CD79B",
        "CD37",
        "CD74",
        "HLA-DRA",
        "CD22"
    ],
    "plasma_cell": [
        "MZB1",
        "JCHAIN",
        "SDC1",
        "XBP1",
        "DERL3",
        "IGKC"
    ],
    "myeloid": [
        "LST1",
        "TYROBP",
        "FCER1G",
        "AIF1",
        "CTSS",
        "LILRB1",
        "CSTA",
        "SAT1"
    ],
    "mast_cell": [
        "TPSAB1",
        "TPSB2",
        "KIT",
        "CPA3",
        "MS4A2",
        "HDC"
    ],
    "endothelial": [
        "PECAM1",
        "VWF",
        "KDR",
        "ESAM",
        "ENG",
        "EMCN",
        "RAMP2",
        "RGCC",
        "PLVAP",
        "CA4"
    ],
    "epithelial_keratin": [
        "EPCAM",
        "TACSTD2",
        "KRT7",
        "KRT8",
        "KRT18",
        "KRT19",
        "MUC1",
        "KRT5",
        "KRT6A",
        "KRT6B",
        "KRT14",
        "KRT17",
        "TP63",
        "SFN",
        "DSG3",
        "CEACAM5",
        "CEACAM6",
        "MSLN",
        "KRT20",
        "MUC13",
        "TFF3"
    ],
    "keratinocyte": [
        "KRT5",
        "KRT14",
        "KRT1",
        "KRT10",
        "DSG1",
        "DSC3",
        "IVL",
        "SFN",
        "KRTDAP"
    ],
    "melanoma_melanocytic": [
        "MLANA",
        "PMEL",
        "TYR",
        "DCT",
        "MITF",
        "SOX10",
        "S100B"
    ],
    "melanoma_dedifferentiated": [
        "AXL",
        "NGFR",
        "SOX9"
    ],
    "fibroblast": [
        "COL1A1",
        "COL1A2",
        "COL3A1",
        "DCN",
        "LUM",
        "PDGFRA",
        "COL6A1",
        "COL6A2",
        "C7"
    ],
    "mural": [
        "RGS5",
        "CSPG4",
        "MCAM",
        "PDGFRB",
        "ACTA2",
        "TAGLN",
        "MYL9",
        "DES"
    ],
    "erythroid": [
        "HBA1",
        "HBA2",
        "HBB",
        "HBD",
        "ALAS2",
        "AHSP",
        "GYPA"
    ]
}

print('scvi-tools:', scvi.__version__)
print('pyUCell:', getattr(uc, '__version__', 'unknown'))
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('A GPU is required to decode ResolVI expression efficiently.')


In [ ]:
# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
PROJECT_ROOT = Path('/host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057')
TMP_ROOT = PROJECT_ROOT / 'tmp'
PIPELINE_ROOT = TMP_ROOT / 'proseg_resolvi_immune_enrichment_v1'
CONFIG_ROOT = PIPELINE_ROOT / '00_config'
RESOLVI_ROOT = PIPELINE_ROOT / '02_resolvi'
SCORED_ROOT = PIPELINE_ROOT / '03_ucell_scored'
ENRICHED_ROOT = PIPELINE_ROOT / '04_immune_endothelial_enriched'
SCORED_ROOT.mkdir(parents=True, exist_ok=True)
ENRICHED_ROOT.mkdir(parents=True, exist_ok=True)

INCLUDE_COLON = True
SECTION_NAMES = [
    sample
    for sample, meta in SAMPLE_INFO.items()
    if INCLUDE_COLON or meta['cancer_type'] != 'colon_cancer'
]

RESOLVI_DECODE_CELL_CHUNK = 1_000
RESOLVI_POSTERIOR_BATCH_SIZE = 512
NORMALIZED_LIBRARY_SIZE = 10_000.0

UCELL_MAX_RANK = 1_500
UCELL_TIES_METHOD = 'average'
UCELL_MISSING_GENES = 'skip'
UCELL_INTERNAL_CHUNK_SIZE = 500
UCELL_N_JOBS = 8
UCELL_DEVICE = 'cpu'   # safest while the ResolVI model remains on the GPU
COMPUTE_RAW_UCELL = False

MIN_DETECTED_IMMUNE_GENES = 2
MIN_DETECTED_ENDOTHELIAL_GENES = 3
MIN_DETECTED_EXCLUSION_GENES = 3

PLOT_DPI = 500
PLOT_MAX_CELLS = 300_000
RANDOM_SEED = 0
H5AD_COMPRESSION = 'lzf'
USE_EXISTING_SCORED = True
OVERWRITE_SCORED = False
CONTINUE_ON_ERROR = True

# Fill this only after inspecting the score plots and threshold grids.
SAMPLE_THRESHOLDS = {
    # 'Screen_30_16': {
    #     'immune_min': 0.10,
    #     'endothelial_min': 0.10,
    #     'exclusion_max': 0.10,
    #     'exclusion_high': 0.20,
    #     'margin_min': 0.00,
    # },
}
RETAIN_AMBIGUOUS_TARGET = True

print('Samples:', SECTION_NAMES)


In [ ]:
# ---------------------------------------------------------------------
# Paths, signatures, and utility functions
# ---------------------------------------------------------------------
def paths_for_sample(sample: str) -> dict[str, Path]:
    resolvi_dir = RESOLVI_ROOT / sample
    scored_dir = SCORED_ROOT / sample
    enriched_dir = ENRICHED_ROOT / sample
    scored_dir.mkdir(parents=True, exist_ok=True)
    enriched_dir.mkdir(parents=True, exist_ok=True)
    return {
        'prepared': resolvi_dir / f'{sample}_resolvi_prepared.h5ad',
        'model': resolvi_dir / 'model',
        'resolvi_final': resolvi_dir / f'{sample}_resolvi_annotated.h5ad',
        'scored_dir': scored_dir,
        'scored': scored_dir / f'{sample}_resolvi_ucell_scored.h5ad',
        'coverage': scored_dir / f'{sample}_signature_coverage.csv',
        'quantiles': scored_dir / f'{sample}_score_quantiles.csv',
        'threshold_grid': scored_dir / f'{sample}_threshold_grid.csv',
        'summary': scored_dir / f'{sample}_ucell_summary.json',
        'enriched': enriched_dir / f'{sample}_immune_endothelial_enriched.h5ad',
        'gate_summary': enriched_dir / f'{sample}_gate_summary.json',
    }


def atomic_write_h5ad(adata: ad.AnnData, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.stem + '.tmp.h5ad')
    if temp.exists():
        temp.unlink()
    adata.write_h5ad(temp, compression=H5AD_COMPRESSION)
    temp.replace(path)


def write_json(payload, path: Path) -> None:
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(payload, indent=2, default=str), encoding='utf-8')
    temp.replace(path)


def load_signatures() -> dict[str, list[str]]:
    path = CONFIG_ROOT / 'signatures_v1.json'
    if path.exists():
        return json.loads(path.read_text(encoding='utf-8'))
    warnings.warn('Signature manifest was not found; using notebook defaults.')
    return DEFAULT_SIGNATURES


def map_signatures_to_var(signatures, var_names):
    lookup = {}
    for name in var_names.astype(str):
        lookup.setdefault(name.upper(), name)
    mapped = {}
    rows = []
    for signature, genes in signatures.items():
        present = []
        missing = []
        for gene in genes:
            match = lookup.get(str(gene).upper())
            if match is None:
                missing.append(gene)
            elif match not in present:
                present.append(match)
        mapped[signature] = present
        rows.append({
            'signature': signature,
            'n_requested': len(genes),
            'n_present': len(present),
            'fraction_present': len(present) / max(len(genes), 1),
            'present_genes': ';'.join(present),
            'missing_genes': ';'.join(missing),
        })
    return mapped, pd.DataFrame(rows)


def signature_columns(signatures, suffix):
    return [f'{name}{suffix}' for name in signatures]


def compute_detected_signature_genes(adata, mapped_signatures):
    X = sp.csr_matrix(adata.X)
    var_index = pd.Series(np.arange(adata.n_vars), index=adata.var_names.astype(str))
    for name, genes in mapped_signatures.items():
        if not genes:
            values = np.zeros(adata.n_obs, dtype=np.int16)
        else:
            idx = var_index.loc[genes].to_numpy(dtype=np.int64)
            values = np.asarray(X[:, idx].getnnz(axis=1)).ravel().astype(np.int16)
        adata.obs[f'n_detected_{name}'] = values


def load_model_and_objects(sample, paths):
    for required in (paths['prepared'], paths['resolvi_final'], paths['model']):
        if not required.exists():
            raise FileNotFoundError(required)
    prepared = ad.read_h5ad(paths['prepared'])
    scored = ad.read_h5ad(paths['resolvi_final'])
    if not prepared.obs_names.equals(scored.obs_names):
        raise ValueError('Prepared and final ResolVI objects have different obs_names/order.')
    if not prepared.var_names.equals(scored.var_names):
        raise ValueError('Prepared and final ResolVI objects have different var_names/order.')
    pyro.clear_param_store()
    model = RESOLVI.load(
        str(paths['model']), adata=prepared, accelerator='gpu', device=0
    )
    return prepared, scored, model


In [ ]:
# ---------------------------------------------------------------------
# Corrected-expression UCell scoring
# ---------------------------------------------------------------------
def compute_resolvi_ucell_scores(prepared, model, mapped_signatures):
    suffix = '_resolvi_UCell'
    columns = signature_columns(mapped_signatures, suffix)
    scores = np.zeros((prepared.n_obs, len(columns)), dtype=np.float32)

    for start in range(0, prepared.n_obs, RESOLVI_DECODE_CELL_CHUNK):
        end = min(start + RESOLVI_DECODE_CELL_CHUNK, prepared.n_obs)
        indices = np.arange(start, end, dtype=np.int64)
        print(
            f'Corrected UCell cells {start:,}:{end:,} / {prepared.n_obs:,}',
            flush=True,
        )
        expression = model.get_normalized_expression(
            adata=prepared,
            indices=indices,
            gene_list=None,
            library_size=NORMALIZED_LIBRARY_SIZE,
            n_samples=1,
            return_mean=True,
            return_numpy=True,
            batch_size=RESOLVI_POSTERIOR_BATCH_SIZE,
        )
        expression = np.asarray(expression, dtype=np.float32)
        chunk = ad.AnnData(
            X=expression,
            obs=pd.DataFrame(index=prepared.obs_names[start:end].copy()),
            var=pd.DataFrame(index=prepared.var_names.copy()),
        )
        uc.compute_ucell_scores(
            chunk,
            signatures=mapped_signatures,
            layer=None,
            max_rank=min(UCELL_MAX_RANK, chunk.n_vars),
            ties_method=UCELL_TIES_METHOD,
            missing_genes=UCELL_MISSING_GENES,
            chunk_size=UCELL_INTERNAL_CHUNK_SIZE,
            suffix=suffix,
            n_jobs=UCELL_N_JOBS,
            device=UCELL_DEVICE,
        )
        scores[start:end, :] = chunk.obs[columns].to_numpy(dtype=np.float32)
        del expression, chunk
        gc.collect()

    return pd.DataFrame(scores, index=prepared.obs_names, columns=columns)


def compute_raw_ucell_scores(scored, mapped_signatures):
    uc.compute_ucell_scores(
        scored,
        signatures=mapped_signatures,
        layer=None,
        max_rank=min(UCELL_MAX_RANK, scored.n_vars),
        ties_method=UCELL_TIES_METHOD,
        missing_genes=UCELL_MISSING_GENES,
        chunk_size=UCELL_INTERNAL_CHUNK_SIZE,
        suffix='_raw_UCell',
        n_jobs=UCELL_N_JOBS,
        device=UCELL_DEVICE,
    )


def add_composite_scores(adata):
    immune = adata.obs['immune_core_resolvi_UCell'].to_numpy(dtype=np.float32)
    endothelial = adata.obs['endothelial_resolvi_UCell'].to_numpy(dtype=np.float32)
    adata.obs['immune_target_score'] = immune
    adata.obs['endothelial_target_score'] = endothelial
    adata.obs['target_score'] = np.maximum(immune, endothelial)
    adata.obs['target_identity_by_score'] = pd.Categorical(
        np.where(immune >= endothelial, 'immune', 'endothelial')
    )

    exclusion_names = [
        'epithelial_keratin', 'keratinocyte', 'melanoma_melanocytic',
        'melanoma_dedifferentiated', 'fibroblast', 'mural', 'erythroid',
    ]
    exclusion_columns = [f'{name}_resolvi_UCell' for name in exclusion_names]
    exclusion_matrix = adata.obs[exclusion_columns].to_numpy(dtype=np.float32)
    adata.obs['exclusion_score'] = exclusion_matrix.max(axis=1)
    adata.obs['exclusion_identity_by_score'] = pd.Categorical(
        np.asarray(exclusion_names, dtype=object)[np.argmax(exclusion_matrix, axis=1)]
    )
    adata.obs['target_margin'] = (
        adata.obs['target_score'].to_numpy(dtype=np.float32)
        - adata.obs['exclusion_score'].to_numpy(dtype=np.float32)
    )

    exclusion_detected = np.column_stack([
        adata.obs[f'n_detected_{name}'].to_numpy(dtype=np.int16)
        for name in exclusion_names
    ])
    adata.obs['n_detected_exclusion_max'] = exclusion_detected.max(axis=1)
    adata.obs['target_multigene_coherent'] = (
        (adata.obs['n_detected_immune_core'].to_numpy(dtype=int) >= MIN_DETECTED_IMMUNE_GENES)
        | (adata.obs['n_detected_endothelial'].to_numpy(dtype=int) >= MIN_DETECTED_ENDOTHELIAL_GENES)
    )


In [ ]:
# ---------------------------------------------------------------------
# Plots and threshold exploration
# ---------------------------------------------------------------------
def plot_indices(n_obs):
    if n_obs <= PLOT_MAX_CELLS:
        return np.arange(n_obs)
    rng = np.random.default_rng(RANDOM_SEED)
    return np.sort(rng.choice(n_obs, PLOT_MAX_CELLS, replace=False))


def get_spatial_xy(adata):
    for key in ('spatial_fullres', 'X_spatial', 'spatial'):
        if key in adata.obsm:
            coords = np.asarray(adata.obsm[key], dtype=np.float32)
            return coords[:, 0], coords[:, 1], key
    raise KeyError('No spatial coordinate matrix found for plotting.')


def save_score_plots(adata, sample, out_dir):
    idx = plot_indices(adata.n_obs)
    for column in ('target_score', 'exclusion_score', 'target_margin', 'immune_target_score', 'endothelial_target_score'):
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.hist(adata.obs[column].to_numpy(dtype=float)[idx], bins=100)
        ax.set_xlabel(column)
        ax.set_ylabel('Number of cells')
        ax.set_title(f'{sample}: {column}')
        fig.tight_layout()
        fig.savefig(out_dir / f'{sample}_{column}_hist_500dpi.png', dpi=PLOT_DPI, bbox_inches='tight')
        plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 7))
    hb = ax.hexbin(
        adata.obs['exclusion_score'].to_numpy(dtype=float)[idx],
        adata.obs['target_score'].to_numpy(dtype=float)[idx],
        gridsize=120,
        bins='log',
        mincnt=1,
    )
    ax.set_xlabel('exclusion_score')
    ax.set_ylabel('target_score')
    ax.set_title(f'{sample}: target versus exclusion')
    fig.colorbar(hb, ax=ax, label='log10(cell count)')
    fig.tight_layout()
    fig.savefig(out_dir / f'{sample}_target_vs_exclusion_hexbin_500dpi.png', dpi=PLOT_DPI, bbox_inches='tight')
    plt.close(fig)

    x, y, coordinate_key = get_spatial_xy(adata)
    for column in ('target_score', 'exclusion_score', 'target_margin', 'immune_target_score', 'endothelial_target_score'):
        fig, ax = plt.subplots(figsize=(9, 9))
        points = ax.scatter(
            x[idx], y[idx], c=adata.obs[column].to_numpy(dtype=float)[idx],
            s=0.5, linewidths=0, rasterized=True,
        )
        ax.set_aspect('equal')
        ax.invert_yaxis()
        ax.set_xlabel(f'{coordinate_key} x')
        ax.set_ylabel(f'{coordinate_key} y')
        ax.set_title(f'{sample}: {column}')
        fig.colorbar(points, ax=ax, label=column)
        fig.tight_layout()
        fig.savefig(out_dir / f'{sample}_{column}_spatial_500dpi.png', dpi=PLOT_DPI, bbox_inches='tight')
        plt.close(fig)


def score_quantile_table(adata):
    columns = [
        column for column in adata.obs.columns
        if column.endswith('_resolvi_UCell')
    ] + ['target_score', 'exclusion_score', 'target_margin']
    quantiles = [0, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1.0]
    rows = []
    for column in columns:
        values = adata.obs[column].to_numpy(dtype=float)
        for q, value in zip(quantiles, np.quantile(values, quantiles)):
            rows.append({'score': column, 'quantile': q, 'value': float(value)})
    return pd.DataFrame(rows)


def threshold_grid(adata):
    target = adata.obs['target_score'].to_numpy(dtype=float)
    exclusion = adata.obs['exclusion_score'].to_numpy(dtype=float)
    coherent = adata.obs['target_multigene_coherent'].to_numpy(dtype=bool)

    target_values = np.unique(np.concatenate([
        np.linspace(0.02, 0.50, 13),
        np.quantile(target, [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.975, 0.99]),
    ]))
    exclusion_values = np.unique(np.concatenate([
        np.linspace(0.02, 0.40, 11),
        np.quantile(exclusion, [0.25, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]),
    ]))

    rows = []
    for target_min in target_values:
        target_evidence = coherent & (target >= target_min)
        for exclusion_max in exclusion_values:
            high_conf = target_evidence & (exclusion <= exclusion_max)
            mixed = target_evidence & (exclusion > exclusion_max)
            rows.append({
                'target_min': float(target_min),
                'exclusion_max': float(exclusion_max),
                'n_high_confidence_target': int(high_conf.sum()),
                'n_target_ambiguous_mixed': int(mixed.sum()),
                'n_target_retained_if_mixed_kept': int((high_conf | mixed).sum()),
                'fraction_high_confidence_target': float(high_conf.mean()),
                'fraction_target_retained_if_mixed_kept': float((high_conf | mixed).mean()),
            })
    return pd.DataFrame(rows)


In [ ]:
# ---------------------------------------------------------------------
# Per-sample scoring runner
# ---------------------------------------------------------------------
def process_scoring_sample(sample):
    paths = paths_for_sample(sample)
    print()
    print('=' * 90)
    print('Corrected UCell sample:', sample)
    print('Output:', paths['scored_dir'])

    if USE_EXISTING_SCORED and not OVERWRITE_SCORED and paths['scored'].exists() and paths['summary'].exists():
        print('Reusing existing scored object.')
        return json.loads(paths['summary'].read_text(encoding='utf-8'))

    started = time.time()
    signatures = load_signatures()
    prepared, scored, model = load_model_and_objects(sample, paths)
    mapped_signatures, coverage = map_signatures_to_var(signatures, prepared.var_names)
    coverage.to_csv(paths['coverage'], index=False)

    corrected_scores = compute_resolvi_ucell_scores(
        prepared, model, mapped_signatures
    )
    scored.obs = scored.obs.drop(columns=corrected_scores.columns, errors='ignore')
    scored.obs = pd.concat([scored.obs, corrected_scores], axis=1)
    compute_detected_signature_genes(scored, mapped_signatures)

    if COMPUTE_RAW_UCELL:
        compute_raw_ucell_scores(scored, mapped_signatures)

    add_composite_scores(scored)
    scored.uns['ucell_signatures'] = signatures
    scored.uns['ucell_scoring'] = {
        'sample': sample,
        'source': 'ResolVI decoded true expression, generated in chunks',
        'normalized_library_size': float(NORMALIZED_LIBRARY_SIZE),
        'ucell_max_rank': int(min(UCELL_MAX_RANK, scored.n_vars)),
        'ucell_missing_genes': UCELL_MISSING_GENES,
        'ucell_device': UCELL_DEVICE,
        'compute_raw_ucell': bool(COMPUTE_RAW_UCELL),
        'min_detected_immune_genes': int(MIN_DETECTED_IMMUNE_GENES),
        'min_detected_endothelial_genes': int(MIN_DETECTED_ENDOTHELIAL_GENES),
        'min_detected_exclusion_genes': int(MIN_DETECTED_EXCLUSION_GENES),
    }

    save_score_plots(scored, sample, paths['scored_dir'])
    quantiles = score_quantile_table(scored)
    quantiles.to_csv(paths['quantiles'], index=False)
    grid = threshold_grid(scored)
    grid.to_csv(paths['threshold_grid'], index=False)

    atomic_write_h5ad(scored, paths['scored'])
    summary = {
        'sample': sample,
        **SAMPLE_INFO[sample],
        'scored_h5ad': str(paths['scored']),
        'signature_coverage_csv': str(paths['coverage']),
        'score_quantiles_csv': str(paths['quantiles']),
        'threshold_grid_csv': str(paths['threshold_grid']),
        'n_cells': int(scored.n_obs),
        'n_genes': int(scored.n_vars),
        'median_target_score': float(scored.obs['target_score'].median()),
        'median_exclusion_score': float(scored.obs['exclusion_score'].median()),
        'fraction_target_multigene_coherent': float(scored.obs['target_multigene_coherent'].mean()),
        'runtime_minutes': (time.time() - started) / 60.0,
    }
    write_json(summary, paths['summary'])
    print('Saved scored object:', paths['scored'])

    del corrected_scores, coverage, quantiles, grid, model, prepared, scored
    pyro.clear_param_store()
    gc.collect()
    torch.cuda.empty_cache()
    return summary


In [ ]:
# ---------------------------------------------------------------------
# Run corrected-expression UCell scoring
# ---------------------------------------------------------------------
scoring_results = {}
scoring_failures = {}
for sample in SECTION_NAMES:
    try:
        scoring_results[sample] = process_scoring_sample(sample)
    except Exception as exc:
        scoring_failures[sample] = repr(exc)
        print(f'[FAILED] {sample}: {type(exc).__name__}: {exc}')
        if not CONTINUE_ON_ERROR:
            raise
    finally:
        pyro.clear_param_store()
        gc.collect()
        torch.cuda.empty_cache()

pd.DataFrame.from_dict(scoring_results, orient='index').to_csv(
    SCORED_ROOT / 'all_samples_ucell_summary.csv', index=False
)
write_json(scoring_failures, SCORED_ROOT / 'all_samples_ucell_failures.json')

print()
print('Completed:', sorted(scoring_results))
print('Failures:', json.dumps(scoring_failures, indent=2))


## Final gate — run only after choosing thresholds

Populate `SAMPLE_THRESHOLDS` in the configuration cell after reviewing each
sample's score distributions, spatial maps, and threshold grid. Then run the
cell below. It does not reload ResolVI or recompute UCell scores.


In [ ]:
# ---------------------------------------------------------------------
# Apply user-selected per-sample gates and save enriched objects
# ---------------------------------------------------------------------
def apply_sample_gate(sample, thresholds):
    paths = paths_for_sample(sample)
    if not paths['scored'].exists():
        raise FileNotFoundError(paths['scored'])
    adata = ad.read_h5ad(paths['scored'])

    required = {
        'immune_min', 'endothelial_min', 'exclusion_max',
        'exclusion_high', 'margin_min',
    }
    missing = required - set(thresholds)
    if missing:
        raise KeyError(f'{sample} thresholds missing keys: {sorted(missing)}')

    immune_evidence = (
        (adata.obs['immune_target_score'].to_numpy(dtype=float) >= thresholds['immune_min'])
        & (adata.obs['n_detected_immune_core'].to_numpy(dtype=int) >= MIN_DETECTED_IMMUNE_GENES)
    )
    endothelial_evidence = (
        (adata.obs['endothelial_target_score'].to_numpy(dtype=float) >= thresholds['endothelial_min'])
        & (adata.obs['n_detected_endothelial'].to_numpy(dtype=int) >= MIN_DETECTED_ENDOTHELIAL_GENES)
    )
    target_evidence = immune_evidence | endothelial_evidence
    exclusion = adata.obs['exclusion_score'].to_numpy(dtype=float)
    margin = adata.obs['target_margin'].to_numpy(dtype=float)

    high_conf = (
        target_evidence
        & (exclusion <= thresholds['exclusion_max'])
        & (margin >= thresholds['margin_min'])
    )
    mixed = target_evidence & ~high_conf
    non_target = (
        ~target_evidence
        & (exclusion >= thresholds['exclusion_high'])
        & (adata.obs['n_detected_exclusion_max'].to_numpy(dtype=int) >= MIN_DETECTED_EXCLUSION_GENES)
    )

    labels = np.full(adata.n_obs, 'unknown_low_information', dtype=object)
    labels[non_target] = 'non_target_high_confidence'
    labels[mixed] = 'target_ambiguous_mixed'
    labels[high_conf] = 'target_high_confidence'
    adata.obs['immune_endothelial_gate'] = pd.Categorical(labels)
    adata.obs['retain_for_immune_endothelial'] = high_conf | (mixed & RETAIN_AMBIGUOUS_TARGET)

    retained = adata[adata.obs['retain_for_immune_endothelial'].to_numpy(dtype=bool)].copy()
    retained.uns['immune_endothelial_gate'] = {
        'sample': sample,
        'thresholds': {key: float(value) for key, value in thresholds.items()},
        'retain_ambiguous_target': bool(RETAIN_AMBIGUOUS_TARGET),
    }
    atomic_write_h5ad(retained, paths['enriched'])

    summary = {
        'sample': sample,
        'thresholds': thresholds,
        'n_total': int(adata.n_obs),
        'n_target_high_confidence': int(high_conf.sum()),
        'n_target_ambiguous_mixed': int(mixed.sum()),
        'n_non_target_high_confidence': int(non_target.sum()),
        'n_retained': int(retained.n_obs),
        'enriched_h5ad': str(paths['enriched']),
    }
    write_json(summary, paths['gate_summary'])
    print(sample, json.dumps(summary, indent=2))
    del adata, retained
    gc.collect()
    return summary


gate_results = {}
if not SAMPLE_THRESHOLDS:
    print('SAMPLE_THRESHOLDS is empty. Review plots and threshold-grid CSVs first.')
else:
    for sample, thresholds in SAMPLE_THRESHOLDS.items():
        gate_results[sample] = apply_sample_gate(sample, thresholds)
    pd.DataFrame.from_dict(gate_results, orient='index').to_csv(
        ENRICHED_ROOT / 'all_samples_gate_summary.csv', index=False
    )
